# Week 6 – Integrative Capstone Project and Evaluation
## Breast Cancer Classification and Patient Pattern Analysis

This notebook combines data acquisition, preprocessing, EDA, unsupervised learning, supervised learning, deep learning, and evaluation into one end-to-end Data Science pipeline.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

np.random.seed(42)
torch.manual_seed(42)
print("PyTorch:", torch.__version__)


## 1. Data Acquisition

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
df = X.copy()
df["target"] = y

print("Shape:", df.shape)
display(df.head())


## 2. Data Cleaning

In [ ]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

numeric = df.drop(columns=["target"])
Q1 = numeric.quantile(0.25)
Q3 = numeric.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = ((numeric < lower) | (numeric > upper)).sum()
print("Total IQR outlier cells:", int(outliers.sum()))


## 3. Exploratory Data Analysis

In [ ]:
print(df.describe().T.head(10))
print("\nClass distribution:")
print(y.value_counts().sort_index())

plt.figure(figsize=(7,4))
y.value_counts().sort_index().plot(kind="bar")
plt.xlabel("Target class")
plt.ylabel("Count")
plt.title("Target Class Distribution")
plt.show()


In [ ]:
corr = numeric.corr()
plt.figure(figsize=(10,8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()


## 4. Unsupervised Learning – K-Means

In [ ]:
cluster_scaler = StandardScaler()
X_scaled = cluster_scaler.fit_transform(numeric)

scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

print(scores)
best_k = max(scores, key=scores.get)
print("Best k:", best_k, "Silhouette:", scores[best_k])

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=clusters)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clusters in PCA Space")
plt.show()

print("Explained variance:", pca.explained_variance_ratio_)


## 5. Supervised Learning – Logistic Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    numeric, y, test_size=0.20, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500, random_state=42))
])
pipeline.fit(X_train, y_train)
lr_pred = pipeline.predict(X_test)

lr_acc = accuracy_score(y_test, lr_pred)
lr_prec, lr_rec, lr_f1, _ = precision_recall_fscore_support(
    y_test, lr_pred, average="weighted", zero_division=0
)
print("Accuracy:", lr_acc)
print("Weighted precision:", lr_prec)
print("Weighted recall:", lr_rec)
print("Weighted F1:", lr_f1)


In [ ]:
cm = confusion_matrix(y_test, lr_pred)
ConfusionMatrixDisplay(cm, display_labels=["malignant","benign"]).plot(values_format="d")
plt.title("Logistic Regression Confusion Matrix")
plt.show()


## 6. Deep Learning – PyTorch MLP

In [ ]:
dl_scaler = StandardScaler()
X_train_s = dl_scaler.fit_transform(X_train).astype(np.float32)
X_test_s = dl_scaler.transform(X_test).astype(np.float32)

train_ds = TensorDataset(
    torch.tensor(X_train_s),
    torch.tensor(y_train.to_numpy()).long()
)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

class BreastCancerMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.net(x)

model = BreastCancerMLP(X_train_s.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
print(model)


In [ ]:
history = {"epoch": [], "loss": [], "test_accuracy": []}

for epoch in range(1, 31):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_test_s)).argmax(1).numpy()
    test_acc = accuracy_score(y_test, pred)

    history["epoch"].append(epoch)
    history["loss"].append(float(loss.item()))
    history["test_accuracy"].append(float(test_acc))

print("Final test accuracy:", history["test_accuracy"][-1])


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history["epoch"], history["loss"])
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("PyTorch Training Loss")
plt.show()

plt.figure(figsize=(8,4))
plt.plot(history["epoch"], history["test_accuracy"])
plt.xlabel("Epoch")
plt.ylabel("Test accuracy")
plt.title("PyTorch Test Accuracy")
plt.show()


In [ ]:
model.eval()
with torch.no_grad():
    dl_pred = model(torch.tensor(X_test_s)).argmax(1).numpy()

dl_acc = accuracy_score(y_test, dl_pred)
dl_prec, dl_rec, dl_f1, _ = precision_recall_fscore_support(
    y_test, dl_pred, average="weighted", zero_division=0
)

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "PyTorch MLP"],
    "Accuracy": [lr_acc, dl_acc],
    "Weighted F1": [lr_f1, dl_f1]
})
comparison


## 7. Integrated Insights

- EDA identifies distributions and relationships before modeling.
- K-Means provides an unsupervised view without using the target label.
- Logistic Regression provides an interpretable supervised baseline.
- The PyTorch MLP adds a non-linear deep-learning approach.
- Evaluation should consider multiple metrics and model complexity, not accuracy alone.


## 8. Recommendations and Reflection

Future work can include hierarchical clustering, DBSCAN, systematic hyperparameter tuning, repeated cross-validation, early stopping, learning-rate scheduling, feature selection, and probability calibration. Any real-world use of a diagnostic dataset would require domain validation and appropriate professional oversight.


## 9. Conclusion
This notebook demonstrates an end-to-end Data Science pipeline from acquisition and preprocessing through EDA, unsupervised learning, supervised learning, deep learning, and evaluation.